In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from openpmd_viewer import OpenPMDTimeSeries
from scipy.constants import epsilon_0, pi
from scipy.special import erf


def w(z):
    return np.exp(-(z**2)) * (1 + erf(1.0j * z))

In [ ]:
sigmaz = 300e-6
sigmax = 100e-6
sigmay = 100e-6
Q = -3.2e-9


def evaluate_E(x, y, z):
    if sigmax == sigmay:
        sigma0 = sigmax
        r = np.sqrt(x**2 + y**2)
        G = np.exp(-z * z / 2.0 / sigmaz**2)
        F = np.exp(-r * r / 2.0 / sigma0**2)
        Er = Q / ((2 * pi) ** (3.0 / 2.0) * epsilon_0 * sigmaz) * G * (1.0 - F) / r
        theta = np.arctan2(y, x)
        Ex = Er * np.cos(theta)
        Ey = Er * np.sin(theta)
        return Ex, Ey
    else:
        E_complex = (
            Q
            / (4 * np.pi * epsilon_0 * sigmaz * np.sqrt((sigmax**2 - sigmay**2)))
            * np.exp(-(z**2) / (2 * sigmaz**2))
            * (
                w((x + 1j * y) / np.sqrt(2 * (sigmax**2 - sigmay**2)))
                - np.exp(-(x**2) / (2 * sigmax**2) - y**2 / (2 * sigmay**2))
                * w(
                    (x * sigmay / sigmax + 1j * y * sigmax / sigmay)
                    / np.sqrt(2 * (sigmax**2 - sigmay**2))
                )
            )
        )
    return E_complex.imag, E_complex.real


theta = np.pi * 0.25

nx = 128
ny = 128
nz = 128

xmax = 6 * sigmax * np.cos(theta) + 3 * sigmaz * np.sin(theta)
zmax = 6 * sigmaz * np.cos(theta) + 3 * sigmax * np.sin(theta)

x = np.linspace(-xmax, +xmax, nx)
y = np.linspace(-4 * sigmay, +4 * sigmay, ny)
z = np.linspace(-zmax, +zmax, nz)

X, Y, Z = np.meshgrid(x, y, z, indexing="ij")

Ex, Ey = evaluate_E(X, Y, Z)
Ez = np.zeros_like(Ex)

R_y = np.array(
    [[np.cos(theta), 0, -np.sin(theta)], [0, 1, 0], [np.sin(theta), 0, np.cos(theta)]]
)

grid = np.stack((X, Y, Z), axis=-1)  # Each point's vector is E[i,j,k,:]

# Apply rotation using tensor dot product
grid_rot = np.tensordot(grid, R_y, axes=([3], [1]))

# Extract rotated components
X_rot = grid_rot[..., 0]
Y_rot = grid_rot[..., 1]
Z_rot = grid_rot[..., 2]

Ex_rot, Ey_rot = evaluate_E(X_rot, Y_rot, Z_rot)
Ez_rot = np.zeros_like(Ex_rot)
E = np.stack((Ex_rot, Ey_rot, Ez_rot), axis=-1)  # Each point's vector is E[i,j,k,:]

# Apply rotation using tensor dot product
E_rot = np.tensordot(E, R_y, axes=([3], [1]))

# Extract rotated components
Ex_rot = E_rot[..., 0]
Ey_rot = E_rot[..., 1]
Ez_rot = E_rot[..., 2]


plt.figure(figsize=(12, 5))
plt.subplot(121)

E_midy = np.sqrt(
    Ex[:, ny // 2, :] ** 2 + Ey[:, ny // 2, :] ** 2 + Ez[:, ny // 2, :] ** 2
)
plt.imshow(
    E_midy,
    extent=(z[0], z[-1], x[0], x[-1]),
    origin="lower",
    aspect="auto",
    vmin=0,
    vmax=3e8,
)
plt.title(r"$\theta = 0^\circ$")
plt.xlabel(r"z")
plt.ylabel(r"x")
plt.colorbar()

plt.subplot(122)
E_midy = np.sqrt(
    Ex_rot[:, ny // 2, :] ** 2 + Ey_rot[:, ny // 2, :] ** 2 + Ez_rot[:, ny // 2, :] ** 2
)
plt.imshow(
    E_midy,
    extent=(z[0], z[-1], x[0], x[-1]),
    origin="lower",
    aspect="auto",
    vmin=0,
    vmax=3e8,
)
plt.plot(x, np.tan(theta) * x, "r--", label="Beam axis")
plt.title(r"$\theta = 45^\circ$")
plt.xlabel(r"z")
plt.ylabel(r"x")
plt.colorbar()

plt.suptitle("theory")
plt.savefig("igf_rotation_theory.png", dpi=300)

In [ ]:
plt.clf()
plt.figure(figsize=(12, 5))

nx = 64
ny = 64
nz = 64

##########################################################
# theta 0

ts = OpenPMDTimeSeries("./diags_theta0/diag2/")
Ex, info = ts.get_field("E", coord="x", iteration=1)
Ey, info = ts.get_field("E", coord="y", iteration=1)
Ez, info = ts.get_field("E", coord="z", iteration=1)

E_midy = np.sqrt(
    Ex[:, ny // 2, :] ** 2 + Ey[:, ny // 2, :] ** 2 + Ez[:, ny // 2, :] ** 2
)

plt.subplot(121)

plt.imshow(
    E_midy.T,
    extent=(info.zmin, info.zmax, info.xmin, info.xmax),
    origin="lower",
    aspect="auto",
    vmin=0,
    vmax=3e8,
)
plt.title(r"$\theta = 0^\circ$")
plt.xlabel(r"z")
plt.ylabel(r"x")
plt.colorbar()


##########################################################
# theta 45

ts = OpenPMDTimeSeries("./diags_theta45/diag2/")
Ex, info = ts.get_field("E", coord="x", iteration=1)
Ey, info = ts.get_field("E", coord="y", iteration=1)
Ez, info = ts.get_field("E", coord="z", iteration=1)

E_midy = np.sqrt(
    Ex[:, ny // 2, :] ** 2 + Ey[:, ny // 2, :] ** 2 + Ez[:, ny // 2, :] ** 2
)

plt.subplot(122)

plt.imshow(
    E_midy.T,
    extent=(info.zmin, info.zmax, info.xmin, info.xmax),
    origin="lower",
    aspect="auto",
    vmin=0,
    vmax=3e8,
)
plt.plot(info.x, np.tan(theta) * info.x, "r--")
plt.title(r"$\theta = 45^\circ$")
plt.xlabel(r"z")
plt.ylabel(r"x")
plt.colorbar()

plt.tight_layout()
plt.suptitle("WarpX new igf branch")

plt.savefig("igf_rotation_newbranch.png", dpi=300)
plt.show()

In [ ]:
plt.clf()
plt.figure(figsize=(12, 5))

nx = 64
ny = 64
nz = 64

##########################################################
# theta 0

ts = OpenPMDTimeSeries("./diags_dev_theta0/diag2/")
Ex, info = ts.get_field("E", coord="x", iteration=1)
Ey, info = ts.get_field("E", coord="y", iteration=1)
Ez, info = ts.get_field("E", coord="z", iteration=1)

E_midy = np.sqrt(
    Ex[:, ny // 2, :] ** 2 + Ey[:, ny // 2, :] ** 2 + Ez[:, ny // 2, :] ** 2
)

plt.subplot(121)

plt.imshow(
    E_midy.T,
    extent=(info.zmin, info.zmax, info.xmin, info.xmax),
    origin="lower",
    aspect="auto",
    vmin=0,
    vmax=3e8,
)
plt.title(r"$\theta = 0^\circ$")
plt.xlabel(r"z")
plt.ylabel(r"x")
plt.colorbar()


##########################################################
# theta 45

ts = OpenPMDTimeSeries("./diags_dev_theta45/diag2/")
Ex, info = ts.get_field("E", coord="x", iteration=1)
Ey, info = ts.get_field("E", coord="y", iteration=1)
Ez, info = ts.get_field("E", coord="z", iteration=1)

E_midy = np.sqrt(
    Ex[:, ny // 2, :] ** 2 + Ey[:, ny // 2, :] ** 2 + Ez[:, ny // 2, :] ** 2
)

plt.subplot(122)

plt.imshow(
    E_midy.T,
    extent=(info.zmin, info.zmax, info.xmin, info.xmax),
    origin="lower",
    aspect="auto",
    vmin=0,
    vmax=3e8,
)
plt.plot(info.x, np.tan(theta) * info.x, "r--")
plt.title(r"$\theta = 45^\circ$")
plt.xlabel(r"z")
plt.ylabel(r"x")
plt.colorbar()

plt.tight_layout()
plt.suptitle("WarpX development branch")

plt.savefig("igf_rotation_warpxdev.png", dpi=300)
plt.show()

In [ ]:
# No rotation

ts = OpenPMDTimeSeries("./diags_norot/diag2/")
Ex, info = ts.get_field("E", coord="x", iteration=1)
Ey, info = ts.get_field("E", coord="y", iteration=1)
print(np.max(np.abs(Ey)) / 1e9, np.max(np.abs(Ex)) / 1e9)

plt.figure(figsize=(10, 5))

plt.subplot(121)
Ex, info = ts.get_field("E", "x", slice_across=["y", "z"], iteration=1, plot=True)
plt.plot(info.x, Ex, "b-", label="WarpX")
Ex_th, Ey_th = evaluate_E(info.x, 0)
plt.plot(info.x, Ex_th, "--", label="Basseti-Erskine formula")
plt.legend(loc=1)
plt.ylim(-2e11, 2e11)
plt.grid()

x_axis = info.x

plt.subplot(122)
Ey, info = ts.get_field("E", "y", slice_across=["x", "z"], iteration=1, plot=True)
plt.plot(info.y, Ey, "b-", label="WarpX")
Ex_th, Ey_th = evaluate_E(0, info.y)
plt.plot(info.y, Ey_th, "--", label="Basseti-Erskine formula")
plt.legend(loc=1)
plt.ylim(-2e11, 2e11)
plt.grid()

y_axis = info.y

In [ ]:
# Rotation

ts = OpenPMDTimeSeries("./diags/diag2/")
plt.figure(figsize=(10, 5))

plt.subplot(111)
rho, info = ts.get_field("rho", iteration=1)
plt.imshow(
    rho[:, 63, :].T,
    extent=[info.zmin, info.zmax, info.xmin, info.xmax],
    origin="lower",
    aspect="auto",
)
plt.grid()

In [ ]:
ts = OpenPMDTimeSeries("./diags/diag2/")

Ex, info = ts.get_field("E", coord="x", iteration=1)
Ey, info = ts.get_field("E", coord="y", iteration=1)
Ez, info = ts.get_field("E", coord="z", iteration=1)

x = info.x
y = info.y
z = info.z

nx = len(x)
ny = len(y)
nz = len(z)

# Rotation angle in radians
theta = 1e-1

X, Y, Z = np.meshgrid(x, y, z, indexing="ij")
Ex_th, Ey_th = evaluate_E(X, Y, Z)
Ez_th = np.zeros_like(Ex_th)

# Rotation matrix around the y-axis
R_y = np.array(
    [[np.cos(theta), 0, np.sin(theta)], [0, 1, 0], [-np.sin(theta), 0, np.cos(theta)]]
)

# Stack the components into a 4D array: shape (Nx, Ny, Nz, 3)
E = np.stack((Ex_th, Ey_th, Ez_th), axis=-1)  # Each point's vector is E[i,j,k,:]

# Apply rotation using tensor dot product
E_rot = np.tensordot(E, R_y, axes=([3], [1]))

# Extract rotated components
Ex_rot = E_rot[..., 0]
Ey_rot = E_rot[..., 1]
Ez_rot = E_rot[..., 2]


plt.figure(figsize=(10, 5))

plt.subplot(121)
plt.imshow(
    np.sqrt(Ex[:, 63, :] ** 2 + Ey[:, 63, :] ** 2 + Ez[:, 63, :] ** 2).T,
    extent=[info.zmin, info.zmax, info.xmin, info.xmax],
    origin="lower",
    aspect="auto",
)
plt.colorbar(label="|E| (V/m)")


plt.subplot(122)
plt.imshow(
    np.sqrt(Ex_rot[:, 63, :] ** 2 + Ey_rot[:, 63, :] ** 2 + Ez_rot[:, 63, :] ** 2).T,
    extent=[info.zmin, info.zmax, info.xmin, info.xmax],
    origin="lower",
    aspect="auto",
)
plt.colorbar(label="|E| (V/m)")


plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.imshow(
    np.sqrt(Ex_rot[:, 63, :] ** 2 + Ey_rot[:, 63, :] ** 2 + Ez_rot[:, 63, :] ** 2).T,
    extent=[info.zmin, info.zmax, info.xmin, info.xmax],
    origin="lower",
    aspect="auto",
)
plt.colorbar(label="|E| (V/m)")
# plt.xlim(-1e-4, 1e-4)
# plt.ylim(-1e-4, 1e-4)
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

plt.subplot(121)


plt.plot(x, Ex_rot[nz // 2, 63, :], "b-", label="WarpX")
Ex_th, Ey_th = evaluate_E(x_axis, 0)
# plt.plot( x_axis, Ex_th, '--', label='Basseti-Erskine formula' )
# plt.legend(loc=1)
plt.grid()

plt.subplot(122)

plt.plot(y, Ey_rot[nz // 2, :, nx // 2], "b-", label="WarpX")
Ex_th, Ey_th = evaluate_E(0, y)
# plt.plot( y_axis, Ey_th, '--', label='Basseti-Erskine formula' )
# plt.legend(loc=1)
plt.grid()